In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W17 出發！目標：把專題資料訓練起來，產出一張能放進簡報的成果卡')
print('資料格式：一個 CSV，前面幾欄是數字特徵，最後一欄是類別答案')
print('影像組直接用 Teachable Machine 訓練與 Demo；本模板給表單／感測／統計類題目用')

# W17 期末專題模板（sklearn 路線，不需要 GPU）
手機開這本請先切成「電腦版網站」（iPhone：網址列 ᴀA → 要求電腦版網站；Android：右上 ⋮ → 電腦版網站），再點每一格左邊的 ▶ 執行。

流程：上傳資料 → 比較三個模型 → 調一個參數 → 看混淆矩陣 → 產出成果卡。還沒有資料也可以先用內建示範資料把整條路走一遍。

**第 0 格**：上傳你們的 CSV（一個檔就好）。還沒有資料就直接跑下一格，會用示範資料。

In [ ]:
#@title 第 0 格：上傳專題資料 CSV（還沒有就直接跑下一格）
from google.colab import files
try:
    up = files.upload()
except Exception:
    up = {}
print('✅ 收到', len(up), '個檔案 → 跑下一格整理資料')

**第 0.5 格**：整理成特徵 X 與答案 y。沒上傳檔案時用鳶尾花示範資料，先把流程走通。

In [ ]:
#@title 第 0.5 格：整理成 X 與 y（沒上傳＝用示範資料）
import pandas as pd                  # ←投影片未含，執行所需
up = globals().get('up', {})         # 沒跑上傳格就當作沒有檔案
if up:
    df = pd.read_csv(list(up)[0])
    X, y = df.iloc[:, :-1], df.iloc[:, -1]
else:
    from sklearn.datasets import load_iris
    X, y = load_iris(return_X_y=True, as_frame=True)
from sklearn.model_selection import train_test_split
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=0)
print('✅ 特徵', X.shape, '；類別：', sorted(set(y)), '→ 往下比較模型')

**第 1 格**：一次比較三個模型。三個分數就是你們的 baseline，抄進紀錄表第一列。

In [ ]:
#@title 第 1 格：一次比較三個模型
from sklearn.model_selection import cross_val_score as cv
from sklearn.linear_model import LogisticRegression as LR
from sklearn.tree import DecisionTreeClassifier as DT
from sklearn.ensemble import RandomForestClassifier as RF
ms = [('LR', LR(max_iter=1000)), ('DT', DT()), ('RF', RF())]
for name, m in ms:
    print(name, round(cv(m, X, y, cv=3).mean(), 3))

**第 2 格**：一次只調一個參數。n 是森林裡有幾棵樹，d 是每棵樹最深幾層。

In [ ]:
#@title 第 2 格：一次只調一個參數
n = 100  #@param {type:"integer"}
d = 5    #@param {type:"integer"}
m = RF(n_estimators=n, max_depth=d, random_state=0)
s = cv(m, X, y, cv=3).mean()
print(f'n={n}　d={d}　交叉驗證分數 = {s:.3f}')

**第 3 格**：測試分數與混淆矩陣。對角線以外最大的那個數字，就是簡報要解釋的重點。

In [ ]:
#@title 第 3 格：測試分數與混淆矩陣
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
m.fit(Xtr, ytr)
p = m.predict(Xte)
print(confusion_matrix(yte, p))
print(classification_report(yte, p))

**成果卡**：填上組別與題目，跑出來的卡片＋上一格的混淆矩陣一起截圖，直接貼進簡報。

In [ ]:
#@title 成果卡：填組別題目，截這張圖放進簡報
TITLE = '第X組 我們的題目'  #@param {type:"string"}

print('=' * 34)
print(' 期末專題成果卡｜AI 導論 115-1')
print(' 組別／題目：', TITLE)
print(f' 模型：RF(n_estimators={n}, max_depth={d})')
print(f' 交叉驗證：{s:.3f}　測試正確率：{m.score(Xte, yte):.3f}')
print('=' * 34)
print('✅ 這張卡＋混淆矩陣一起截圖，貼進簡報')

**選用**：把模型存到雲端硬碟。W18 現場 Demo 直接載入這個檔，絕對不要重新訓練——台下等進度條是最傷分的畫面。

In [ ]:
#@title 選用：存模型到雲端硬碟（Demo 當天直接載入）
from google.colab import drive   # ←投影片未含，執行所需
drive.mount('/content/drive')
import joblib, os
os.makedirs('/content/drive/MyDrive/AI115專題', exist_ok=True)  # ←投影片未含
out = '/content/drive/MyDrive/AI115專題/model.pkl'
joblib.dump(m, out)
def predict_one(row):
    return m.predict([row])[0]
print('✅ 已存檔，Demo 載入這個檔再呼叫 predict_one 就好')